In [2]:
# ==============================================================================
# 1. INSTALL DEPENDENCIES & INITIALIZE
# ==============================================================================
!pip install -q pyspark

import time
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, count

print("1. Initializing Spark Session...")
spark = (
    SparkSession.builder
    .appName("Flight_DQ_Partitioning_Demo")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.driver.memory", "8g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

# ==============================================================================
# 2. INGESTION & NATIVE PYSPARK DATA QUALITY CHECKS
# ==============================================================================
print("2. Ingesting ~6 Million Row Flight Dataset...")
RAW_PATH = "/kaggle/input/datasets/usdot/flight-delays/flights.csv"
OUTPUT_UNPARTITIONED = "/kaggle/working/flights_unpartitioned.parquet"
OUTPUT_PARTITIONED = "/kaggle/working/flights_partitioned.parquet"

# Read raw CSV
raw_df = spark.read.csv(RAW_PATH, header=True, inferSchema=True)

print("3. Running Native Distributed Data Quality Validation...")
# Evaluate all rules in a single distributed pass over the data
dq_metrics = raw_df.select(
    # Rule 1: Tail numbers cannot be completely null (check ratio)
    (count(when(col("TAIL_NUMBER").isNull(), 1)) / count("*")).alias("tail_num_null_ratio"),
    
    # Rule 2: Scheduled departure times must be valid military time (0 to 2359)
    count(when((col("SCHEDULED_DEPARTURE") < 0) | (col("SCHEDULED_DEPARTURE") > 2359), 1)).alias("invalid_time_count"),
    
    # Rule 3: Cancelled flag must be boolean (0 or 1)
    count(when(~col("CANCELLED").isin([0, 1]), 1)).alias("invalid_cancelled_count")
).collect()[0]

# Verify expectations
failed_expectations = []
if dq_metrics["tail_num_null_ratio"] > 0.01:
    failed_expectations.append(f"Tail Number nulls ({dq_metrics['tail_num_null_ratio']:.1%}) exceeded 1% threshold.")
if dq_metrics["invalid_time_count"] > 0:
    failed_expectations.append(f"Found {dq_metrics['invalid_time_count']} invalid departure times.")
if dq_metrics["invalid_cancelled_count"] > 0:
    failed_expectations.append(f"Found {dq_metrics['invalid_cancelled_count']} invalid cancelled flags.")

if failed_expectations:
    print(f"   -> [WARNING] Data Quality checks failed!")
    for issue in failed_expectations:
        print(f"      - {issue}")
else:
    print("   -> [SUCCESS] All Data Quality checks passed. Safe to process.")

# ==============================================================================
# 3. PROCESSING & PARTITIONED WRITES
# ==============================================================================
print("\n4. Writing Unpartitioned Data (Simulating Bad Practice)...")
(
    raw_df.write
    .mode("overwrite")
    .parquet(OUTPUT_UNPARTITIONED)
)

print("5. Writing Partitioned Data (Best Practice: Year/Month)...")
# Partitioning physically separates the files into folders like YEAR=2015/MONTH=1/
(
    raw_df.write
    .mode("overwrite")
    .partitionBy("YEAR", "MONTH")
    .parquet(OUTPUT_PARTITIONED)
)

# ==============================================================================
# 4. BENCHMARKING: UNPARTITIONED VS PARTITIONED (THE PROOF)
# ==============================================================================
print("\n6. Running Performance Benchmark (Simulating a BI Dashboard Query)...")
print("   Query: 'Count all Delta Air Lines (DL) flights delayed by >30 mins in July'")

# Read the written Parquet files back into memory
df_unpartitioned = spark.read.parquet(OUTPUT_UNPARTITIONED)
df_partitioned = spark.read.parquet(OUTPUT_PARTITIONED)

# Benchmark 1: Full Table Scan (Unpartitioned)
start_time = time.time()
unpartitioned_result = (
    df_unpartitioned
    .filter((col("YEAR") == 2015) & (col("MONTH") == 7) & (col("AIRLINE") == 'DL') & (col("DEPARTURE_DELAY") > 30))
    .count()
)
unpartitioned_time = time.time() - start_time

# Benchmark 2: Partition Pruning (Partitioned)
# Spark engine looks at the query, ignores 11 months of data, and only scans the July folder
start_time = time.time()
partitioned_result = (
    df_partitioned
    .filter((col("YEAR") == 2015) & (col("MONTH") == 7) & (col("AIRLINE") == 'DL') & (col("DEPARTURE_DELAY") > 30))
    .count()
)
partitioned_time = time.time() - start_time

print("\n=== BENCHMARK RESULTS ===")
print(f"Result Count: {partitioned_result} flights")
print(f"Unpartitioned Query Time: {unpartitioned_time:.3f} seconds (Full Table Scan)")
print(f"Partitioned Query Time:   {partitioned_time:.3f} seconds (Partition Pruning)")

speedup = unpartitioned_time / partitioned_time if partitioned_time > 0 else 0
print(f"Performance Gain: Partitioned query was {speedup:.1f}x faster.")
print("=========================\n")

1. Initializing Spark Session...
2. Ingesting ~6 Million Row Flight Dataset...


3. Running Native Distributed Data Quality Validation...


   -> [SUCCESS] All Data Quality checks passed. Safe to process.

4. Writing Unpartitioned Data (Simulating Bad Practice)...


5. Writing Partitioned Data (Best Practice: Year/Month)...



6. Running Performance Benchmark (Simulating a BI Dashboard Query)...
   Query: 'Count all Delta Air Lines (DL) flights delayed by >30 mins in July'

=== BENCHMARK RESULTS ===
Result Count: 7463 flights
Unpartitioned Query Time: 0.978 seconds (Full Table Scan)
Partitioned Query Time:   0.400 seconds (Partition Pruning)
Performance Gain: Partitioned query was 2.4x faster.

